# RQ3 — Preset BigAug

Run `unet3d_bigaug_pli0` du tableau d'ablation : UNet MONAI 3D, patches de 96³, **pli 0** de `splits.json`, **graine 2026**,
40 000 itérations (lots de 4 patches), identiques pour toutes les techniques. Durée estimée sur une T4 : 4 à 6 h
(garde-fou à 10 h d'entraînement, session limitée à 12 h).

**Avant de lancer** : accélérateur *GPU T4*, Internet activé, dataset privé `atlas-c1-processed` ajouté en entrée.
Notebook à garder **privé** (accord d'utilisation ATLAS). *Save Version → Save & Run All* pour une exécution en arrière-plan.

In [ ]:
# Code du projet (dépôt public) et dépendances absentes de l'image Kaggle.
import hashlib, json, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/working/PROJET_AVC")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "adaptation-entrainement", "https://github.com/gadhgadhi-yosra/PROJET_AVC.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.1", "torchio==1.2.1", "nibabel==5.4.2",
                "pyarrow"], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Données C1 : liens vers les dossiers de sujets, quelle que soit l'archive d'origine.
import zipfile
INPUT = Path("/kaggle/input/atlas-c1-processed")
DATA = Path("/kaggle/working/processed")
DATA.mkdir(exist_ok=True)
sources = [INPUT]
if not any(INPUT.rglob("qc.json")):  # archives restées compressées : extraction sur le disque de travail
    extracted = Path("/kaggle/working/c1_extrait")
    for archive in sorted(INPUT.rglob("processed_partie*.zip")):
        zipfile.ZipFile(archive).extractall(extracted)
    sources = [extracted]
for qc in (path for source in sources for path in source.rglob("qc.json")):
    link = DATA / qc.parent.name
    if not link.exists():
        link.symlink_to(qc.parent, target_is_directory=True)
table = DATA / "subjects.parquet"
if not table.exists():
    table.symlink_to(next(INPUT.rglob("subjects.parquet")))
print(sum(1 for _ in DATA.glob("*/qc.json")), "sujets prétraités")
# Même découpage que le code : splits.json du dataset identique à celui du dépôt.
digest = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert digest(next(INPUT.rglob("splits.json"))) == digest(REPO / "splits.json"), "splits.json différent du dépôt"

In [ ]:
# Entraînement, validation sur le pli, puis évaluation sur le test ID du pli et le test OOD.
# Session interrompue : rattacher la sortie précédente, copier last.pt dans /kaggle/working/runs/unet3d_bigaug_pli0/ et ajouter --resume.
!cd {REPO} && python -m adaptation.train --technique bigaug --fold 0 --seed 2026 --data-root {DATA} --output /kaggle/working/runs --num-workers 4

In [ ]:
# Résumé et fichiers à récupérer (onglet Output) ; aucune image de patient n'est écrite.
for path in sorted(Path("/kaggle/working/runs/unet3d_bigaug_pli0").glob("resume*.json")):
    print(path.name, json.dumps(json.loads(path.read_text()), indent=2, ensure_ascii=False))
!ls -la /kaggle/working/runs/unet3d_bigaug_pli0